In [ ]:
# Rankine-Hugoniot Jump Conditions and Shock Cut (paper figures)
#
# Reads the data cached by notebooks/rh_conds/rh_conds_mime.ipynb and reproduces the
# two figures used in the previous paper's rh_conds.ipynb - "rh" and "shock_cut" - here
# named "rh_mime" and "shock_cut_mime". This notebook has no graphet/TristanRun
# dependency - it only needs the saved NetCDF file.

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12


def save_polished_fig(name, dpi=300, metadata=None):
    plt.savefig(f"/scratch/gpfs/ar3676/plots/{name}", bbox_inches='tight', dpi=dpi, metadata=metadata)


findex = 250  # must match the findex used when the data was saved

data = xr.open_dataset(f"../rh_conds/rh_conds_data_{findex}.nc")

xlim = data.attrs['xlim']
boundary = data.attrs['boundary']

print(f"Loaded RH data for findex={findex}: theta = {data.attrs['angle_in_deg']:.2f} deg "
      f"({data.attrs['theta_method']}), steps = {list(data.attrs['my_steps'])}")


In [ ]:
# RH Conditions ("rh_mime")

def plot_squant(axis, quantity):
    axis.plot(data.x, data[quantity])


fig, axes = plt.subplots(3, 2, figsize=(8, 4), sharex=True)
ax1, ax2, ax3, ax4, ax5, ax6 = axes[0, 0], axes[1, 0], axes[2, 0], axes[0, 1], axes[1, 1], axes[2, 1]

plt.subplots_adjust(hspace=0, wspace=0.2)

eqn_fsize = 18

# Mass

plot_squant(ax1, 'mass_flux')
ax1.text(0.45, 0.10, r"$\rho v_n$", transform=ax1.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Pressure

plot_squant(ax2, 'pressure')
ax2.set_ylim(0, 1.2)
ax2.text(0.35, 0.10, r"$\rho v_n^2 + P_g + P_B$", transform=ax2.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Energy

plot_squant(ax3, 'energy')
energy_expression = r'$\rho v_n \left(\frac{v^2}{2} + \frac{\gamma}{\gamma - 1} \frac{P_g}{\rho} + \frac{B^2}{4 \pi \rho}\right) - \frac{B_n (\mathbf{v} \cdot \mathbf{B})}{4 \pi}$'
ax3.text(0.15, 0.10, energy_expression, transform=ax3.transAxes, fontsize=12, verticalalignment='bottom')

# B_n

plot_squant(ax4, 'B_n')
ax4.text(0.45, 0.10, '$B_n$', transform=ax4.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Poynting Flux

plot_squant(ax5, 'poynting')
ax5.text(0.35, 0.10, r'$\rho v_n v_t - \frac{B_n B_t} {4 \pi}$', transform=ax5.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Electric Field

plot_squant(ax6, 'E_ideal')
ax6.text(0.35, 0.10, '$B_n v_t - v_n B_t$', transform=ax6.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

for ax_row in axes:
    for ax in ax_row:
        ax.set_xlim(xlim / 2, xlim * 0.95)
        ax.axvline(boundary, color='green', linestyle='--')

for ax in [ax3, ax6]:
    ax.set_xlabel("$x$ [$c / \omega_p$]")

plt.show()

save_polished_fig("rh_mime")


In [ ]:
# Shock Cut ("shock_cut_mime")

plt.rcParams.update({'font.size': 18})

max_jz = data.jz.sel(x=slice(xlim / 2, xlim)).max()

plt.figure()
plt.axvline(boundary, color='green', linestyle='--')
plt.plot(data.x, data.dens, label="$n / n_0$")
plt.plot(data.x, data.pn, label="$P_g / P_{B0}$")
plt.plot(data.x, data.pb, label="$P_B / P_{B0}$")
plt.plot(data.x, data.jz / max_jz, label=r"$j_z / j_S$")

plt.xlim(xlim / 2, xlim * 0.95)
plt.xlabel("$x$ [$c / \omega_p$]")
plt.legend()
plt.show()
plt.tight_layout()

plt.rcParams.update({'font.size': 12})

save_polished_fig("shock_cut_mime")
